# models3.ipynb — Structured Argumentation Graph

## Overview

This notebook builds a **Graph Attention Network (GAT)** for fake-news detection on the
[ISOT dataset](https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets).
It is the third iteration in this series; see `models2.ipynb` for the previous version.

Each news article is converted into a **Structured Argumentation Graph**: a typed graph
where nodes represent sentences (labelled by rhetorical role) and edges encode semantic
similarity and logical relations (entailment / contradiction / neutral) both within the
article and across externally retrieved articles covering the same event.

---

## Core redesign from models2

### What was wrong
`models2` searched using entity queries extracted from the article body. This produced
irrelevant results (emojipedia, dndbeyond, tacomaworld) and left **262 / 500 graphs with
zero evidence nodes** — graphs that all look structurally identical and give the model
nothing to learn from.

### What changes

**1. Headline-based search** *(O(1) per article, not O(n_claims))*  
Headlines are written to be precise and findable. Searching the headline directly
retrieves articles covering the *same event* from different publishers — exactly the
cross-source comparison we want.

**2. Sentence role classification** *(zero-shot NLI)*  
Each sentence is labelled as one of: `claim`, `evidence`, `analysis`, `background`.
Misinformation manipulates the *analysis* layer while keeping evidence plausible — so
this distinction is the key signal that was missing.

**3. Cross-source analysis comparison**  
The target article's analysis sentences are compared via NLI to analysis sentences from
retrieved articles covering the same topic. Analysis entailed by multiple independent
sources is credible; analysis that contradicts them is a fake signal.

**4. Typed graph structure**  
Nodes carry role labels. Edges connect: intra-article (sentence↔sentence) and
cross-source (target analysis ↔ retrieved analysis). Node features include role type and
NLI relation counts broken down by role.

---

**Setup:** place `Fake.csv` and `True.csv` from the ISOT dataset in a `data/` folder.

In [1]:
import os, json, hashlib, math, time, sqlite3
from contextlib import contextmanager
from urllib.parse import urlparse
from datetime import datetime, timezone

import nltk
try:
    nltk.download('punkt_tab', quiet=True)
except Exception:
    nltk.download('punkt', quiet=True)

import spacy
nlp = spacy.load('en_core_web_sm')

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
import numpy as np
import networkx as nx
from nltk.tokenize import sent_tokenize

ENCODER   = SentenceTransformer('all-MiniLM-L6-v2')
NLI_MODEL = CrossEncoder('cross-encoder/nli-deberta-v3-small')

# NLI label indices for nli-deberta-v3-small
NLI_CONTRADICTION = 0
NLI_ENTAILMENT    = 1
NLI_NEUTRAL       = 2

# Edge type constants
EDGE_NEUTRAL       = 0
EDGE_ENTAILMENT    = 1
EDGE_CONTRADICTION = 2

# Sentence role constants
ROLE_CLAIM      = 0
ROLE_EVIDENCE   = 1
ROLE_ANALYSIS   = 2
ROLE_BACKGROUND = 3
ROLE_NAMES      = ['claim', 'evidence', 'analysis', 'background']

# Node feature dimension — expanded to include role encoding
FEATURE_DIM = 20

print('Setup complete.')

Setup complete.


## Step 1: Load ISOT Dataset

The ISOT Fake News Dataset contains ~23 000 fake articles (from unreliable sources
flagged by fact-checking organisations) and ~21 000 real articles (from Reuters.com).
We shuffle, binary-encode the label, and strip the Reuters dateline from real articles
to prevent the model from learning a trivial formatting heuristic.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

fake = pd.read_csv('data/Fake.csv')
real = pd.read_csv('data/True.csv')
fake['label'] = 'fake'
real['label'] = 'real'

df = pd.concat([fake, real]).sample(frac=1, random_state=42).reset_index(drop=True)
df['label_binary'] = (df['label'] == 'fake').astype(int)

# Strip Reuters dateline from real articles to prevent trivial leakage
df['text'] = df['text'].str.replace(
    r'^[A-Z\s,]+\([^)]+\)\s*-\s*', '', regex=True
).str.strip()

print(f'Total: {len(df)} | Balance: {df["label_binary"].value_counts().to_dict()}')
df[['title', 'text', 'label']].head(2)

Total: 44898 | Balance: {1: 23481, 0: 21417}


,title,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",fake
1,Trump drops Steve Bannon from National Securit...,U.S. President Donald Trump removed his chief ...,real


## Step 2: Sentence Role Classification

Each sentence is classified as **claim**, **evidence**, **analysis**, or **background**
using zero-shot NLI. Rather than fine-tuning a dedicated classifier, we run the NLI
model against four defining hypotheses and pick the highest-scoring one.

This is the key structural addition over `models2`. Misinformation rarely fabricates raw
events — it manipulates the *analysis* layer: presenting selective evidence, drawing
unsupported conclusions, or framing neutral facts with loaded interpretation. Making this
distinction explicit in the graph gives the model the right signal to learn from.

**Why not fine-tune a classifier?** Zero-shot NLI generalises across topics without any
labelled role data. A fine-tuned classifier would need a role-labelled news corpus that
doesn't readily exist and would risk overfitting to surface patterns of specific
publications.

In [3]:
# Defining hypotheses for each role — written to activate the NLI model's
# entailment signal when the premise (sentence) matches the role description.
ROLE_HYPOTHESES = [
    "This sentence makes a specific factual assertion or claim.",      # ROLE_CLAIM
    "This sentence presents data, quotes, or cited supporting evidence.", # ROLE_EVIDENCE
    "This sentence provides interpretation, opinion, or analysis.",    # ROLE_ANALYSIS
    "This sentence provides background context or general information.", # ROLE_BACKGROUND
]


def classify_sentence_roles(sentences: list[str]) -> list[int]:
    """
    Classify each sentence into one of 4 roles using zero-shot NLI.

    For each sentence we run NLI against all 4 hypotheses in a single
    batch call. The hypothesis with the highest entailment score wins.

    Batching all sentences × all hypotheses in one predict() call is
    much faster than calling predict() once per sentence.
    """
    if not sentences:
        return []

    # Build all (sentence, hypothesis) pairs
    pairs = [
        (sent, hyp)
        for sent in sentences
        for hyp in ROLE_HYPOTHESES
    ]

    # Batch NLI inference
    logits    = NLI_MODEL.predict(pairs)                          # (n_sent * 4, 3)
    probs     = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    ent_probs = probs[:, NLI_ENTAILMENT]                          # entailment probability only

    # Reshape to (n_sentences, 4) and argmax per sentence
    ent_matrix = ent_probs.reshape(len(sentences), len(ROLE_HYPOTHESES))
    roles      = np.argmax(ent_matrix, axis=1).tolist()
    return roles


# Sanity check
test_sents = [
    "President Trump signed an executive order on Friday.",
    "According to documents obtained by the Times, the memo was dated March 3.",
    "This represents a fundamental shift in American foreign policy.",
    "The conflict between the two nations began in 1948."
]
roles = classify_sentence_roles(test_sents)
for s, r in zip(test_sents, roles):
    print(f'[{ROLE_NAMES[r]:<12}] {s}')

[claim       ] President Trump signed an executive order on Friday.
[background  ] According to documents obtained by the Times, the memo was dated March 3.
[claim       ] This represents a fundamental shift in American foreign policy.
[analysis    ] The conflict between the two nations began in 1948.


## Step 3: Headline Search

We search using the **article headline** instead of entity queries from the body text.

**Why headlines work better:**
- Headlines are purpose-written to be precise and findable — they are the journalist's best distillation of the article's core claim
- A headline search retrieves articles covering the *same event* from different publishers, which is exactly the cross-source comparison we want
- Entity queries from body text match individual entities that may appear in completely unrelated contexts (hence emojipedia and dndbeyond appearing as evidence)

This reduces from O(n_claims) search calls to O(1) per article.

In [5]:
from duckduckgo_search import DDGS

CACHE_DIR = 'search_cache_v3'  # separate cache from models2 to avoid stale results


def _cache_path(query: str) -> str:
    os.makedirs(CACHE_DIR, exist_ok=True)
    return os.path.join(CACHE_DIR, hashlib.md5(query.encode()).hexdigest() + '.json')


def ddg_search(query: str, num_results: int = 10) -> list[dict]:
    cache_file = _cache_path(query)
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=num_results):
            results.append({
                'link':    r.get('href', ''),
                'title':   r.get('title', ''),
                'snippet': r.get('body', '')
            })
    with open(cache_file, 'w') as f:
        json.dump(results, f)
    time.sleep(0.5)
    return results


def clean_headline(title: str) -> str:
    """
    Strip publication suffixes from headlines before searching.
    e.g. 'Biden signs bill | Reuters' → 'Biden signs bill'
    These suffixes bias results back toward the same publisher.
    """
    for sep in [' | ', ' - ', ' – ', ' — ']:
        if sep in title:
            title = title.split(sep)[0]
    return title.strip()


def collect_evidence_by_headline(title: str, k: int = 10) -> list[dict]:
    """
    Search using the article headline to find coverage of the same event
    from different publishers. Returns raw search result dicts.
    """
    query = clean_headline(title)
    if not query or len(query) < 10:
        return []
    try:
        return ddg_search(query, num_results=k)
    except Exception as e:
        print(f'  Search failed for "{query[:60]}": {e}')
        return []

## Step 4: Source Credibility Database

We maintain a lightweight per-domain **Bayesian credibility score** backed by SQLite.
Each domain starts with a Beta(2, 2) prior (score = 0.5 — no information).

- **Model updates**: when the GAT classifies an article with high confidence, every
  domain that contributed an evidence node receives a fractional update weighted by
  the document's relevance score and the model's confidence.
- **User updates**: explicit human labels can be injected with weight 1.0 (vs 0.3 for
  model updates) to let ground-truth feedback dominate.

The credibility score is blended with the DBSCAN consensus score when ranking retrieved
documents, so high-credibility sources get proportionally more edge weight in the graph.

In [6]:
DB_PATH      = 'source_credibility_v3.db'
MODEL_WEIGHT = 0.3
USER_WEIGHT  = 1.0


@contextmanager
def get_db():
    conn = sqlite3.connect(DB_PATH)
    try:
        yield conn
        conn.commit()
    finally:
        conn.close()


def init_db():
    with get_db() as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS sources (
                domain        TEXT PRIMARY KEY,
                alpha         REAL DEFAULT 2.0,
                beta          REAL DEFAULT 2.0,
                model_updates INTEGER DEFAULT 0,
                user_updates  INTEGER DEFAULT 0,
                last_updated  TEXT
            )''')
        conn.execute('''
            CREATE TABLE IF NOT EXISTS credibility_log (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                domain      TEXT,
                signal_type TEXT,
                signal      TEXT,
                confidence  REAL,
                timestamp   TEXT
            )''')

init_db()


def extract_domain(url: str) -> str:
    return urlparse(url).netloc.replace('www.', '')


def get_credibility(domain: str) -> float:
    with get_db() as conn:
        row = conn.execute(
            'SELECT alpha, beta FROM sources WHERE domain = ?', (domain,)
        ).fetchone()
    return (row[0] / (row[0] + row[1])) if row else 0.5


def update_credibility(domain: str, signal: str, signal_type: str, confidence: float = 1.0):
    assert signal in ('real', 'fake') and signal_type in ('model', 'user')
    weight = (MODEL_WEIGHT if signal_type == 'model' else USER_WEIGHT) * confidence
    now    = datetime.now(timezone.utc).isoformat()  # fixed deprecation warning
    with get_db() as conn:
        conn.execute(
            'INSERT OR IGNORE INTO sources (domain,alpha,beta,last_updated) VALUES(?,2.0,2.0,?)',
            (domain, now)
        )
        col        = 'alpha' if signal == 'real' else 'beta'
        update_col = 'model_updates' if signal_type == 'model' else 'user_updates'
        conn.execute(
            f'UPDATE sources SET {col}={col}+?,{update_col}={update_col}+1,last_updated=? WHERE domain=?',
            (weight, now, domain)
        )
        conn.execute(
            'INSERT INTO credibility_log(domain,signal_type,signal,confidence,timestamp) VALUES(?,?,?,?,?)',
            (domain, signal_type, signal, confidence, now)
        )


def bulk_update_from_prediction(scored_docs, prediction: float,
                                 model_confidence: float,
                                 confidence_threshold: float = 0.1) -> int:
    if model_confidence < confidence_threshold:
        return 0
    signal = 'fake' if prediction > 0.5 else 'real'
    n = 0
    for doc, doc_score in scored_docs:
        url = doc.get('link', '')
        if url:
            update_credibility(extract_domain(url), signal, 'model',
                               confidence=model_confidence * doc_score)
            n += 1
    return n


print('DB ready at', DB_PATH)

DB ready at source_credibility_v3.db


## Step 5: Structured Argumentation Graph

The graph now has two kinds of nodes and three kinds of edges:

**Nodes:**
- `input_*` — sentences from the target article, labeled by role (claim/evidence/analysis/background)
- `ext_*` — sentences from retrieved articles, labeled by role

**Edges:**
- **Intra-article** (input↔input): cosine similarity > threshold, then NLI-typed
- **Cross-source analysis** (input_analysis↔ext_analysis): NLI comparison between target article's analysis and retrieved articles' analysis — this is the core new signal
- **Cross-source evidence** (input_claim↔ext_evidence): NLI comparison between target claims and retrieved evidence

Cross-source edges are only drawn between matching role pairs to avoid noise. Comparing a background sentence to an analysis sentence from another source produces meaningless NLI scores.

In [7]:
def extract_and_embed(text: str) -> tuple[list[str], np.ndarray]:
    """Tokenize text into sentences and embed all at once."""
    sentences = [s.strip() for s in sent_tokenize(text) if len(s.strip()) > 10]
    if not sentences:
        return [], np.array([])
    embeddings = ENCODER.encode(sentences, batch_size=32, show_progress_bar=False)
    return sentences, embeddings


def nli_batch(pairs: list[tuple[str, str]]) -> np.ndarray:
    """Run NLI on a list of (premise, hypothesis) pairs. Returns (n, 3) prob array."""
    if not pairs:
        return np.array([])
    logits = NLI_MODEL.predict(pairs)
    return np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)


def edge_type_from_probs(probs_row: np.ndarray) -> tuple[int, float]:
    label = int(np.argmax(probs_row))
    etype = EDGE_CONTRADICTION if label == NLI_CONTRADICTION else \
            EDGE_ENTAILMENT    if label == NLI_ENTAILMENT    else EDGE_NEUTRAL
    return etype, float(probs_row[label])


def build_article_graph(text: str,
                         sim_threshold: float = 0.60,
                         use_nli: bool = True) -> tuple[nx.Graph, list[str], list[int]]:
    """
    Build the base knowledge graph from the article text.
    Returns the graph, the list of sentences, and their role labels.
    """
    sentences, embeddings = extract_and_embed(text)
    G = nx.Graph()
    if not sentences:
        return G, [], []

    roles = classify_sentence_roles(sentences)

    for i, (sent, emb, role) in enumerate(zip(sentences, embeddings, roles)):
        G.add_node(i, text=sent, embedding=emb, source='input',
                   role=role, weight=1.0)

    sim_matrix = cosine_similarity(embeddings)
    candidate_pairs = [
        (i, j) for i in range(len(sentences))
        for j in range(i + 1, len(sentences))
        if sim_matrix[i, j] > sim_threshold
    ]

    if use_nli and candidate_pairs:
        pairs_text = [(sentences[i], sentences[j]) for i, j in candidate_pairs]
        probs_all  = nli_batch(pairs_text)
        for (i, j), probs in zip(candidate_pairs, probs_all):
            etype, conf = edge_type_from_probs(probs)
            G.add_edge(i, j, similarity=float(sim_matrix[i, j]),
                       edge_type=etype, nli_confidence=conf,
                       edge_source='intra')
    else:
        for i, j in candidate_pairs:
            G.add_edge(i, j, similarity=float(sim_matrix[i, j]),
                       edge_type=EDGE_NEUTRAL, nli_confidence=0.5,
                       edge_source='intra')

    return G, sentences, roles


def augment_with_cross_source(G: nx.Graph,
                               article_sentences: list[str],
                               article_roles: list[int],
                               retrieved_docs: list[dict],
                               sim_threshold: float = 0.55,
                               score_threshold: float = 0.2,
                               use_nli: bool = True) -> tuple[nx.Graph, list[tuple[dict, float]]]:
    """
    Augment the graph with sentences from retrieved articles.

    Cross-source edges are only drawn between semantically similar sentences
    of *matching roles*:
      - input analysis ↔ ext analysis  (is our interpretation corroborated?)
      - input claim    ↔ ext evidence  (is our assertion backed externally?)
      - input claim    ↔ ext claim     (do other sources assert the same thing?)

    Comparing background to analysis across sources produces noise.

    Returns the augmented graph and a list of (doc, score) for the DB update.
    """
    if not retrieved_docs:
        return G, []

    # Score documents by DBSCAN consensus + credibility
    scored_docs = score_documents(retrieved_docs)

    # Get input node info for cross-source comparison
    input_nodes = [(nid, data) for nid, data in G.nodes(data=True)
                   if data.get('source') == 'input']

    # Index article sentences by role for efficient lookup
    article_embs = np.array([data['embedding'] for _, data in input_nodes])

    next_id = max(G.nodes()) + 1 if G.nodes() else 0
    new_nodes, new_edges = [], []

    for doc, doc_score in scored_docs:
        if doc_score < score_threshold or not doc.get('snippet'):
            continue

        ext_sents, ext_embs = extract_and_embed(doc['snippet'])
        if not ext_sents:
            continue

        ext_roles = classify_sentence_roles(ext_sents)

        # Compute similarity between all external and all input sentences at once
        sim_matrix = cosine_similarity(ext_embs, article_embs)  # (n_ext, n_input)

        # Collect candidate cross-source pairs
        cross_pairs_text  = []
        cross_pairs_index = []  # (ext_idx, input_node_idx)

        for ext_i, (ext_sent, ext_role) in enumerate(zip(ext_sents, ext_roles)):
            for inp_j, (inp_nid, inp_data) in enumerate(input_nodes):
                if sim_matrix[ext_i, inp_j] < sim_threshold:
                    continue
                inp_role = inp_data['role']

                # Only compare matching or complementary role pairs
                valid = (
                    (ext_role == ROLE_ANALYSIS   and inp_role == ROLE_ANALYSIS)  or
                    (ext_role == ROLE_EVIDENCE   and inp_role == ROLE_CLAIM)     or
                    (ext_role == ROLE_CLAIM      and inp_role == ROLE_CLAIM)     or
                    (ext_role == ROLE_CLAIM      and inp_role == ROLE_ANALYSIS)
                )
                if valid:
                    cross_pairs_text.append((ext_sent, inp_data['text']))
                    cross_pairs_index.append((ext_i, inp_j, inp_nid,
                                              float(sim_matrix[ext_i, inp_j]),
                                              ext_role))

        # Batch NLI on all cross-source pairs for this document
        if use_nli and cross_pairs_text:
            cross_probs = nli_batch(cross_pairs_text)
        else:
            cross_probs = np.full((len(cross_pairs_text), 3), 1/3)

        # Add external sentences as nodes and draw edges
        ext_node_ids = {}
        for pi, (ext_i, inp_j, inp_nid, sim, ext_role) in enumerate(cross_pairs_index):
            # Add external node if not already added for this doc
            if ext_i not in ext_node_ids:
                nid = next_id
                next_id += 1
                ext_node_ids[ext_i] = nid
                new_nodes.append((
                    nid, ext_sents[ext_i], ext_embs[ext_i], doc_score, ext_role
                ))

            ext_nid = ext_node_ids[ext_i]
            etype, conf = edge_type_from_probs(cross_probs[pi])
            new_edges.append((
                ext_nid, inp_nid, sim, doc_score, etype, conf, 'cross_source'
            ))

    # Commit all changes to G
    for nid, text, emb, score, role in new_nodes:
        G.add_node(nid, text=text, embedding=emb, source='evidence',
                   role=role, weight=score)
    for nid, inp_nid, sim, score, etype, conf, esrc in new_edges:
        G.add_edge(nid, inp_nid, similarity=sim, evidence_weight=score,
                   edge_type=etype, nli_confidence=conf, edge_source=esrc)

    return G, scored_docs


def score_documents(documents: list[dict],
                    credibility_alpha: float = 0.6,
                    eps: float = 0.3,
                    min_samples: int = 2) -> list[tuple[dict, float]]:
    """DBSCAN consensus clustering + domain credibility blending."""
    snippets = [doc.get('snippet', '') for doc in documents]
    if not snippets:
        return []
    embeddings  = ENCODER.encode(snippets, batch_size=32, show_progress_bar=False)
    sim_matrix  = cosine_similarity(embeddings)
    dist_matrix = np.clip(1.0 - sim_matrix, 0, 2).astype(np.float64)
    labels      = DBSCAN(eps=eps, min_samples=min_samples,
                         metric='precomputed').fit_predict(dist_matrix)
    unique      = [l for l in set(labels) if l != -1]
    cons_label  = max(unique, key=lambda l: (labels == l).sum()) if unique else None

    scored = []
    for i, doc in enumerate(documents):
        if cons_label is not None:
            if labels[i] == cons_label:
                members = np.where(labels == cons_label)[0]
                cs      = float(sim_matrix[i, members].mean())
            elif labels[i] == -1:
                cs = 0.1
            else:
                cs = float(sim_matrix[i].mean()) * 0.5
        else:
            cs = float((sim_matrix[i].sum() - 1.0) / max(len(documents) - 1, 1))

        domain   = extract_domain(doc.get('link', ''))
        combined = credibility_alpha * cs + (1 - credibility_alpha) * get_credibility(domain)
        scored.append((doc, float(combined)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

## Step 6: Node Features (20-dimensional)

Each node in the graph is described by a 20-dimensional hand-crafted feature vector.
Features 0–13 were present in `models2`; features 14–19 are new in this version.

| Index | Feature | Description |
|-------|---------|-------------|
| 0 | `n_evidence_nbrs` | Number of external (evidence) neighbours |
| 1 | `n_input_nbrs` | Number of intra-article neighbours |
| 2 | `mean_ev_weight` | Mean document score of evidence neighbours |
| 3 | `max_ev_weight` | Max document score of evidence neighbours |
| 4 | `mean_sim` | Mean edge cosine similarity |
| 5 | `ev_ratio` | Fraction of neighbours that are external |
| 6 | `is_evidence` | 1 if this node is from an external source |
| 7 | `node_weight` | Document credibility / relevance score |
| 8 | `n_entailing` | Count of entailment edges |
| 9 | `n_contradicting` | Count of contradiction edges |
| 10 | `ent_wsum` | Weighted entailment sum (weight × confidence) |
| 11 | `cont_wsum` | Weighted contradiction sum |
| 12 | `ent_ratio` | Entailment fraction of evidence neighbours |
| 13 | `cont_ratio` | Contradiction fraction of evidence neighbours |
| 14–17 | `role_onehot` | One-hot role: claim / evidence / analysis / background |
| 18 | `cross_ent_w` | Cross-source entailment weight (corroboration signal) |
| 19 | `cross_cont_w` | Cross-source contradiction weight (dispute signal) |

In [8]:
import torch
from torch_geometric.data import Data


def compute_node_features(G: nx.Graph) -> np.ndarray:
    feature_list = []
    for node_id, data in G.nodes(data=True):
        neighbors     = list(G.neighbors(node_id))
        ev_nbrs       = [n for n in neighbors if G.nodes[n].get('source') == 'evidence']
        in_nbrs       = [n for n in neighbors if G.nodes[n].get('source') == 'input']
        ev_weights    = [G.edges[node_id, n].get('evidence_weight', 0.0) for n in ev_nbrs]
        all_sims      = [G.edges[node_id, n].get('similarity', 0.0) for n in neighbors]

        # NLI breakdown — all edges
        entailing     = [(n, G.edges[node_id, n]) for n in ev_nbrs
                         if G.edges[node_id, n].get('edge_type') == EDGE_ENTAILMENT]
        contradicting = [(n, G.edges[node_id, n]) for n in ev_nbrs
                         if G.edges[node_id, n].get('edge_type') == EDGE_CONTRADICTION]

        ent_wsum  = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                        for _, e in entailing)
        cont_wsum = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                        for _, e in contradicting)
        n_ev = max(len(ev_nbrs), 1)

        # Cross-source specific breakdown
        cross_nbrs = [n for n in ev_nbrs
                      if G.edges[node_id, n].get('edge_source') == 'cross_source']
        cross_ent  = [(n, G.edges[node_id, n]) for n in cross_nbrs
                      if G.edges[node_id, n].get('edge_type') == EDGE_ENTAILMENT]
        cross_cont = [(n, G.edges[node_id, n]) for n in cross_nbrs
                      if G.edges[node_id, n].get('edge_type') == EDGE_CONTRADICTION]

        cross_ent_w  = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                           for _, e in cross_ent)
        cross_cont_w = sum(e.get('evidence_weight', 0) * e.get('nli_confidence', 0.5)
                           for _, e in cross_cont)

        # Role one-hot
        role    = data.get('role', ROLE_BACKGROUND)
        role_oh = [1.0 if role == r else 0.0 for r in range(4)]

        feature_list.append([
            float(len(ev_nbrs)),                                       # 0
            float(len(in_nbrs)),                                       # 1
            float(np.mean(ev_weights) if ev_weights else 0.0),         # 2
            float(np.max(ev_weights)  if ev_weights else 0.0),         # 3
            float(np.mean(all_sims)   if all_sims   else 0.0),         # 4
            float(len(ev_nbrs) / max(len(neighbors), 1)),              # 5
            1.0 if data.get('source') == 'evidence' else 0.0,          # 6
            float(data.get('weight', 1.0)),                            # 7
            float(len(entailing)),                                     # 8
            float(len(contradicting)),                                 # 9
            ent_wsum,                                                  # 10
            cont_wsum,                                                 # 11
            float(len(entailing)    / n_ev),                           # 12
            float(len(contradicting) / n_ev),                          # 13
            *role_oh,                                                  # 14-17
            cross_ent_w,                                               # 18
            cross_cont_w,                                              # 19
        ])

    arr     = np.array(feature_list, dtype=np.float32)
    col_max = arr.max(axis=0)
    col_max[col_max == 0] = 1
    return arr / col_max


def graph_to_pyg(G: nx.Graph, label: int | None = None) -> Data:
    node_ids = list(G.nodes())
    id_map   = {nid: i for i, nid in enumerate(node_ids)}
    x        = torch.tensor(compute_node_features(G), dtype=torch.float)

    if G.edges():
        edges      = [(id_map[u], id_map[v]) for u, v in G.edges()]
        edge_index = torch.tensor(edges, dtype=torch.long).T
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_feats = []
        for u, v in G.edges():
            ed  = G.edges[u, v]
            et  = ed.get('edge_type', EDGE_NEUTRAL)
            esrc = 1.0 if ed.get('edge_source') == 'cross_source' else 0.0
            oh  = [1.0 if et == k else 0.0 for k in range(3)]
            edge_feats.append(oh + [
                ed.get('similarity', 0.0),
                ed.get('nli_confidence', 0.5),
                ed.get('evidence_weight', 0.0),
                esrc  # whether this is a cross-source edge
            ])
        ef_tensor = torch.tensor(edge_feats * 2, dtype=torch.float)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        ef_tensor  = torch.zeros((0, 7), dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=ef_tensor)
    if label is not None:
        data.y = torch.tensor([label], dtype=torch.float)
    return data

## Step 7: Graph Attention Network (GAT)

The classifier is a 4-layer **Graph Attention Network** with the following design choices:

- **Input projection**: a linear layer maps the 20-dim node features into the hidden
  space before any message passing. This decouples feature scale from hidden dimension.
- **4 × GATConv layers** with multi-head attention (4 heads) and skip connections.
  The first three layers use `concat=True` (outputs are concatenated across heads);
  the final layer uses `concat=False` (outputs are averaged) to control the growth of
  the hidden dimension.
- **Jumping Knowledge (JK) aggregation**: representations from all four layers are
  concatenated (`xjk`), so the readout can draw on local *and* long-range structure
  simultaneously.
- **Global pooling**: both `global_mean_pool` and `global_max_pool` are applied to
  `xjk` and concatenated. Mean captures average node behaviour; max captures the most
  extreme signal anywhere in the graph.
- **MLP classifier**: a 3-layer MLP with BatchNorm, ReLU, and Dropout maps the pooled
  graph representation to a single logit (binary cross-entropy loss).
- **Training**: Adam with cosine annealing LR schedule, gradient clipping, and early
  stopping on validation loss.

In [9]:
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool
from torch_geometric.loader import DataLoader


class FakeNewsGAT(torch.nn.Module):
    def __init__(self, input_dim=FEATURE_DIM, hidden_dim=64, n_heads=4, dropout=0.4):
        super().__init__()
        self.dropout    = dropout
        self.input_proj = torch.nn.Linear(input_dim, hidden_dim)

        self.conv1 = GATConv(hidden_dim,     hidden_dim,     heads=n_heads, concat=True,  dropout=dropout)
        self.lin1  = torch.nn.Linear(hidden_dim * n_heads, hidden_dim)
        self.bn1   = torch.nn.BatchNorm1d(hidden_dim)

        self.conv2 = GATConv(hidden_dim,     hidden_dim * 2, heads=n_heads, concat=True,  dropout=dropout)
        self.lin2  = torch.nn.Linear(hidden_dim * 2 * n_heads, hidden_dim * 2)
        self.bn2   = torch.nn.BatchNorm1d(hidden_dim * 2)

        self.conv3 = GATConv(hidden_dim * 2, hidden_dim * 2, heads=n_heads, concat=True,  dropout=dropout)
        self.lin3  = torch.nn.Linear(hidden_dim * 2 * n_heads, hidden_dim * 2)
        self.bn3   = torch.nn.BatchNorm1d(hidden_dim * 2)

        self.conv4 = GATConv(hidden_dim * 2, hidden_dim,     heads=n_heads, concat=False, dropout=dropout)
        self.bn4   = torch.nn.BatchNorm1d(hidden_dim)

        pool_dim = (hidden_dim + hidden_dim*2 + hidden_dim*2 + hidden_dim) * 2
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(pool_dim, 256), torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(), torch.nn.Dropout(dropout),
            torch.nn.Linear(256, 64),  torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(), torch.nn.Dropout(dropout),
            torch.nn.Linear(64, 1)
        )

    def forward(self, x, edge_index, batch):
        x    = F.relu(self.input_proj(x))
        x    = F.dropout(x, p=self.dropout, training=self.training)
        x1   = self.bn1(F.relu(self.lin1(self.conv1(x,  edge_index)))) + x   # residual: hidden → hidden
        x2   = self.bn2(F.relu(self.lin2(self.conv2(x1, edge_index))))        # no residual: dim doubles
        x3   = self.bn3(F.relu(self.lin3(self.conv3(x2, edge_index)))) + x2  # residual: hidden*2 → hidden*2
        x4   = self.bn4(F.relu(self.conv4(x3, edge_index))) + x3[:, :x1.shape[1]]  # residual: hidden*2 → hidden
        xjk  = torch.cat([x1, x2, x3, x4], dim=1)
        xp   = torch.cat([global_mean_pool(xjk, batch), global_max_pool(xjk, batch)], dim=1)
        return self.mlp(xp)


def train_gat(train_data, val_data, epochs=150, lr=5e-4, patience=20):
    model     = FakeNewsGAT()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = torch.nn.BCEWithLogitsLoss()
    t_loader  = DataLoader(train_data, batch_size=16, shuffle=True)
    v_loader  = DataLoader(val_data,   batch_size=16)

    best_val, no_improve, best_state = float('inf'), 0, None
    for epoch in range(epochs):
        model.train()
        t_loss = 0
        for b in t_loader:
            optimizer.zero_grad()
            out  = model(b.x, b.edge_index, b.batch).squeeze()
            loss = criterion(out, b.y.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item()
        scheduler.step()

        model.eval()
        v_loss = 0
        with torch.no_grad():
            for b in v_loader:
                v_loss += criterion(
                    model(b.x, b.edge_index, b.batch).squeeze(), b.y.squeeze()
                ).item()

        avg_t, avg_v = t_loss / len(t_loader), v_loss / len(v_loader)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch {epoch+1:3d}  train={avg_t:.4f}  val={avg_v:.4f}  '
                  f'lr={scheduler.get_last_lr()[0]:.2e}')

        if avg_v < best_val:
            best_val, no_improve = avg_v, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return model


def evaluate_model(model, dataset):
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for b in DataLoader(dataset, batch_size=16):
            probs = torch.sigmoid(model(b.x, b.edge_index, b.batch).squeeze())
            preds.extend(probs.tolist())
            labels.extend(b.y.squeeze().tolist())
    binary = [1 if p > 0.5 else 0 for p in preds]
    return {
        'accuracy': accuracy_score(labels, binary),
        'f1':       f1_score(labels, binary, zero_division=0),
        'auc':      roc_auc_score(labels, preds)
    }, preds


def save_model(model, path):
    torch.save(model.state_dict(), path)
    print(f'Saved to {path}')

def load_model(path):
    m = FakeNewsGAT()
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    return m

## Step 8: Build Dataset

For each article we:
1. Tokenise the body into sentences and classify their roles (NLI pass).
2. Search DuckDuckGo with the article headline and retrieve up to 10 results.
3. Score the retrieved documents by DBSCAN consensus + domain credibility.
4. Augment the intra-article graph with cross-source edges (role-matched NLI pairs).
5. Compute the 20-dim node feature matrix and convert to a PyG `Data` object.

> **Note:** this build is slower per article than `models2` because role classification
> adds one full NLI pass per article. However it produces far fewer empty graphs since
> headline search returns topically relevant results for almost every article.

In [10]:
from tqdm import tqdm


def build_dataset(df, text_col='text', title_col='title',
                  label_col='label_binary', sample=None, use_nli=True):
    if sample:
        df = df.groupby(label_col, group_keys=False).apply(
            lambda g: g.sample(sample // 2, random_state=42)
        ).reset_index(drop=True)

    pyg_data, all_scored = [], []
    n_aug, n_fall = 0, 0
    role_counts = {r: 0 for r in ROLE_NAMES}

    for _, row in tqdm(df.iterrows(), total=len(df), desc='Building graphs'):
        text  = row[text_col]
        title = row.get(title_col, '')
        label = int(row[label_col])

        # Build base article graph with role-labeled nodes
        G, article_sents, article_roles = build_article_graph(text, use_nli=use_nli)
        if not article_sents:
            all_scored.append([])
            continue

        # Track role distribution for diagnostics
        for r in article_roles:
            role_counts[ROLE_NAMES[r]] += 1

        # Headline search + cross-source augmentation
        scored_docs = []
        try:
            docs = collect_evidence_by_headline(title, k=10)
            if docs:
                G, scored_docs = augment_with_cross_source(
                    G, article_sents, article_roles, docs, use_nli=use_nli
                )
                n_aug += 1
            else:
                n_fall += 1
        except Exception as e:
            print(f'  Augmentation failed: {e}')
            n_fall += 1

        pyg_data.append(graph_to_pyg(G, label=label))
        all_scored.append(scored_docs)

    total_roles = sum(role_counts.values())
    print(f'\nBuilt {len(pyg_data)} | Augmented: {n_aug} | Fallback: {n_fall}')
    print('Role distribution:', {k: f'{v/total_roles:.1%}' for k, v in role_counts.items()})
    return pyg_data, all_scored


pyg_dataset, all_scored_docs = build_dataset(df, sample=500)

C:\Users\bhada\AppData\Local\Temp\ipykernel_18644\2900861423.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(label_col, group_keys=False).apply(
Building graphs:   0%|          | 0/500 [00:00<?, ?it/s]C:\Users\bhada\AppData\Local\Temp\ipykernel_18644\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:   0%|          | 1/500 [00:05<47:11,  5.67s/it]C:\Users\bhada\AppData\Local\Temp\ipykernel_18644\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:   0

  Search failed for "RNC Chief Strategist Has A FULL MELTDOWN On CNN After Hearin": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=RNC+Chief+Strategist+Has+A+FULL+MELTDOWN+On+CNN+After+Hearing+FBI+Won%E2%80%99t+Indict+Hillary+%28VIDEO%29)', 'https://www.bing.com/search?q=RNC+Chief+Strategist+Has+A+FULL+MELTDOWN+On+CNN+After+Hearing+FBI+Won%E2%80%99t+Indict+Hillary+%28VIDEO%29')


C:\Users\bhada\AppData\Local\Temp\ipykernel_18644\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  81%|████████▏ | 407/500 [31:56<18:06, 11.68s/it]

  Search failed for "Trump Bribed An Author To Hide These Facts About How Cruelly": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=Trump+Bribed+An+Author+To+Hide+These+Facts+About+How+Cruelly+He+Divorced+One+Of+His+Wives)', 'https://www.bing.com/search?q=Trump+Bribed+An+Author+To+Hide+These+Facts+About+How+Cruelly+He+Divorced+One+Of+His+Wives')


C:\Users\bhada\AppData\Local\Temp\ipykernel_18644\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs:  82%|████████▏ | 408/500 [32:04<16:22, 10.68s/it]

...

  Search failed for "Trump STRANDED, In Full Panic As ANOTHER Performer Drops Out": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=Trump+STRANDED%2C+In+Full+Panic+As+ANOTHER+Performer+Drops+Out+Of+Inauguration+%28DETAILS%29)', 'https://www.bing.com/search?q=Trump+STRANDED%2C+In+Full+Panic+As+ANOTHER+Performer+Drops+Out+Of+Inauguration+%28DETAILS%29')


C:\Users\bhada\AppData\Local\Temp\ipykernel_18644\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs: 100%|█████████▉| 499/500 [37:42<00:02,  2.03s/it]

  Search failed for "SHOCKING REPORT: 50% of Babies in 24 States Born via Medicai": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=SHOCKING+REPORT%3A+50%25+of+Babies+in+24+States+Born+via+Medicaid%E2%80%A6Is+Your+State+on+the+List%3F)', 'https://www.bing.com/search?q=SHOCKING+REPORT%3A+50%25+of+Babies+in+24+States+Born+via+Medicaid%E2%80%A6Is+Your+State+on+the+List%3F')


C:\Users\bhada\AppData\Local\Temp\ipykernel_18644\13514029.py:17: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Building graphs: 100%|██████████| 500/500 [37:46<00:00,  4.53s/it]

  Search failed for "BERNIE SUES To Allow 17 Year Olds To Vote": https://www.bing.com/search ConnectError: ('error sending request for url (https://www.bing.com/search?q=BERNIE+SUES+To+Allow+17+Year+Olds+To+Vote)', 'https://www.bing.com/search?q=BERNIE+SUES+To+Allow+17+Year+Olds+To+Vote')

Built 493 | Augmented: 87 | Fallback: 406
Role distribution: {'claim': '75.2%', 'evidence': '0.1%', 'analysis': '9.4%', 'background': '15.3%'}


## Step 9: Train, Evaluate, and Update Credibility DB

We do a 70 / 15 / 15 train/val/test split, train the GAT with early stopping, evaluate
on the held-out test set, and then run inference on the full dataset to update the source
credibility database with the model's predictions.

In [13]:
indices             = list(range(len(pyg_dataset)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_data = [pyg_dataset[i] for i in train_idx]
val_data   = [pyg_dataset[i] for i in val_idx]
test_data  = [pyg_dataset[i] for i in test_idx]
print(f'Split: {len(train_data)} train | {len(val_data)} val | {len(test_data)} test')

model = train_gat(train_data, val_data, epochs=150, patience=20)

metrics, test_probs = evaluate_model(model, test_data)
print(f'\nTest Results:')
print(f'  Accuracy : {metrics["accuracy"]:.3f}')
print(f'  F1       : {metrics["f1"]:.3f}')
print(f'  AUC-ROC  : {metrics["auc"]:.3f}')

# Update credibility DB
model.eval()
n_db = 0
with torch.no_grad():
    for batch, scored in zip(DataLoader(pyg_dataset, batch_size=1), all_scored_docs):
        if not scored:
            continue
        prob = torch.sigmoid(model(batch.x, batch.edge_index, batch.batch).squeeze()).item()
        conf = abs(prob - 0.5) * 2
        n_db += bulk_update_from_prediction(scored, prob, conf, confidence_threshold=0.1)

print(f'\nCredibility DB: {n_db} entries updated.')
save_model(model, 'fake_news_gat_v3.pt')

Split: 345 train | 74 val | 74 test


c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
c:\Users\bhada\anaconda3\Lib\site-packages\torch_geometric\inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params

Epoch  10  train=0.5339  val=0.6192  lr=4.95e-04
Epoch  20  train=0.5132  val=0.6269  lr=4.78e-04
Early stopping at epoch 28

Test Results:
  Accuracy : 0.743
  F1       : 0.716
  AUC-ROC  : 0.859

Credibility DB: 481 entries updated.
Saved to fake_news_gat_v3.pt


In [14]:
with get_db() as conn:
    rows = conn.execute(
        'SELECT domain, alpha, beta, model_updates, user_updates '
        'FROM sources ORDER BY model_updates + user_updates DESC LIMIT 25'
    ).fetchall()

print(f'{"Domain":<40} {"Score":>6} {"Signals":>8} {"Model":>7} {"User":>6}')
print('-' * 70)
for domain, alpha, beta, m, u in rows:
    score = alpha / (alpha + beta)
    n     = int(alpha + beta - 4)
    print(f'{domain:<40} {score:>6.3f} {n:>8} {m:>7} {u:>6}')

Domain                                    Score  Signals   Model   User
----------------------------------------------------------------------
en.wikipedia.org                          0.602        2      56      0
en.m.wikipedia.org                        0.582        2      38      0
britannica.com                            0.547        1      36      0
youtube.com                               0.493        0      29      0
merriam-webster.com                       0.437        1      26      0
apnews.com                                0.539        1      24      0
nbcnews.com                               0.536        1      24      0
nytimes.com                               0.542        1      24      0
whitehouse.gov                            0.557        1      22      0
wasserfaelle-krimml.at                    0.419        0      20      0
cbsnews.com                               0.512        1      19      0
usatoday.com                              0.534        0      19 

## Architecture Improvement Proposals

These are concrete changes that could improve the model, in rough order of
expected impact.

---

### 1. Replace `GATConv` with `GATv2Conv` *(high impact, trivial change)*

The original GAT attention mechanism has a theoretical limitation: it computes attention
weights before combining the query and key representations, making it equivalent to a
static (input-independent) attention in certain graph structures. GATv2 fixes this by
applying the non-linearity *after* concatenating the node representations, making
attention genuinely dynamic.

```python
# Change in imports:
from torch_geometric.nn import GATv2Conv  # replaces GATConv

# Change in __init__: replace GATConv(...) → GATv2Conv(...)
# The API is identical; no other changes needed.
self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=n_heads, concat=True, dropout=dropout)
```

---

### 2. Feed edge features into attention *(medium impact)*

The graph carries rich edge attributes (edge type, NLI confidence, similarity, cross-source flag)
but `GATConv` / `GATv2Conv` can only use them if you pass `edge_dim`. Currently they are
computed but ignored during message passing.

```python
EDGE_DIM = 7  # matches the edge_attr dimension in graph_to_pyg()

self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=n_heads,
                       concat=True, dropout=dropout, edge_dim=EDGE_DIM)
# ... same for conv2, conv3, conv4

# In forward():
x1 = self.bn1(F.relu(self.lin1(self.conv1(x, edge_index, edge_attr=edge_attr)))) + x
```

You would also need to add `edge_attr` as a parameter to `forward()` and pass `b.edge_attr`
in the training loop.

---

### 3. Use a richer sentence encoder *(medium impact)*

`all-MiniLM-L6-v2` is fast but relatively weak for semantic nuance. Upgrading to
`all-mpnet-base-v2` (same API, 420 MB vs 80 MB) consistently gives +2–4 points on
STS benchmarks and would produce better embeddings for both the graph edges and
NLI role classification.

```python
ENCODER = SentenceTransformer('all-mpnet-base-v2')
```

Alternatively, `BAAI/bge-small-en-v1.5` is only marginally larger than MiniLM but
significantly stronger.

---

### 4. Replace column-max normalisation with Z-score standardisation *(low-medium impact)*

The current normalisation divides each feature column by its maximum value. This is
sensitive to outliers (one very large value collapses all others toward 0) and doesn't
centre the features, which can slow down learning.

```python
from sklearn.preprocessing import StandardScaler

# In build_dataset, after collecting all pyg_data:
all_x = torch.cat([d.x for d in pyg_data], dim=0).numpy()
scaler = StandardScaler().fit(all_x)

for d in pyg_data:
    d.x = torch.tensor(scaler.transform(d.x.numpy()), dtype=torch.float)
```

Save the scaler alongside the model weights so you can normalise at inference time.

---

### 5. Add a heterogeneous graph formulation *(high impact, more work)*

Right now, `input` and `evidence` nodes are structurally identical in the model — only
feature 6 (`is_evidence`) distinguishes them. PyG's `HeteroData` lets you define
separate embedding spaces and message-passing weights for different node/edge types,
which is a much more principled way to handle this.

```python
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

data = HeteroData()
data['input'].x    = input_features
data['evidence'].x = evidence_features
data['input',  'similar_to', 'input'].edge_index    = intra_edges
data['input',  'supported_by', 'evidence'].edge_index = cross_edges
data['input',  'contradicted_by', 'evidence'].edge_index = contra_edges
```

This is the most architecturally significant change but also the largest refactor.

---

### 6. Add a role-prediction auxiliary loss *(low-medium impact)*

If you have any ground-truth role labels (or can create a small labelled sample with
`classify_sentence_roles` as a noisy teacher), you can add a node-level auxiliary loss
that forces the model to learn role-aware representations. Multi-task learning often
improves the primary task even when the auxiliary task is noisy.

```python
# In forward(), add a branch from the node embeddings before pooling:
role_logits = self.role_head(xjk)  # shape (n_nodes, 4)

# Training loss:
loss = bce_loss(graph_logit, label) + 0.1 * ce_loss(role_logits, node_role_labels)
```